In [ ]:
import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import statsmodels.api as sm

from mcDETECT.utils import *
from mcDETECT.downstream import *

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [ ]:
# Data paths
data_path_wt = "../../data/MERSCOPE_WT_1/processed_data/"
data_path_ad = "../../data/MERSCOPE_AD_1/processed_data/"
data_path = "../../output/MERSCOPE_WT_AD_comparison/"
output_path = "../../output/benchmark/benchmark_ambient/"

In [ ]:
# # Helper function (same as 5_neuropil_subdomains_data.py)
# def fill_spot_expression(spots: "sc.AnnData", transcripts: pd.DataFrame, grid_len: float = 50.0, gene_col: str = "target", spots_x_col: str = "global_x", spots_y_col: str = "global_y", transcripts_x_col = "global_x", transcripts_y_col = "global_y", batch_col: str = "batch", inplace: bool = True, assign_to: str = "X", tx_chunk: int = 65_536, spot_chunk: int = 512) -> "sc.AnnData":
    
#     """Assign each transcript to spot tiles by exact axis-aligned containment (half-open intervals).

#     Tile for spot center (cx, cy) is [cx - h, cx + h) x [cy - h, cy + h) with h = grid_len / 2, matching
#     subdivide_spots. Counts are per (gene, spot); batch must match between transcript and spot rows.

#     This avoids shared histogram bins: each spot gets only transcripts inside its own square. If two tiles
#     overlap, a transcript in the overlap is counted toward every spot whose tile contains it.

#     Parameters
#     ----------
#     assign_to : str
#         "X" to write ``spots.X``, or a layer name to write ``spots.layers[assign_to]``.
#     tx_chunk, spot_chunk : int
#         Chunk sizes to bound peak memory for the boolean overlap array (spots_chunk x transcripts_chunk).
#     """
    
#     if not inplace:
#         spots = spots.copy()
#     half = float(grid_len) / 2.0
#     genes = list(spots.var_names)
#     gene_to_idx = {g: i for i, g in enumerate(genes)}
#     cx = spots.obs[spots_x_col].to_numpy(dtype=float)
#     cy = spots.obs[spots_y_col].to_numpy(dtype=float)
#     bsp = spots.obs[batch_col].astype(str).to_numpy()
#     n_obs, n_vars = spots.n_obs, spots.n_vars
#     X = np.zeros((n_obs, n_vars), dtype=np.float64)
#     batches = transcripts[batch_col].unique()

#     for idx, g in enumerate(genes):
#         gi = gene_to_idx[g]
#         tg = transcripts[transcripts[gene_col] == g]
#         if len(tg) == 0:
#             continue
#         for b in batches:
#             bs = str(b)
#             t_b = tg[tg[batch_col].astype(str) == bs]
#             if len(t_b) == 0:
#                 continue
#             rows = np.flatnonzero(bsp == bs)
#             if rows.size == 0:
#                 continue
#             tx_all = t_b[transcripts_x_col].to_numpy(dtype=float)
#             ty_all = t_b[transcripts_y_col].to_numpy(dtype=float)
#             cx_b = cx[rows]
#             cy_b = cy[rows]
#             counts = np.zeros(rows.size, dtype=np.float64)
#             for r0 in range(0, rows.size, spot_chunk):
#                 r1 = min(r0 + spot_chunk, rows.size)
#                 cx_c = cx_b[r0:r1, None]
#                 cy_c = cy_b[r0:r1, None]
#                 acc = np.zeros(r1 - r0, dtype=np.float64)
#                 for t0 in range(0, tx_all.shape[0], tx_chunk):
#                     t1 = min(t0 + tx_chunk, tx_all.shape[0])
#                     xc = tx_all[t0:t1][None, :]
#                     yc = ty_all[t0:t1][None, :]
#                     acc += np.sum(
#                         (xc >= cx_c - half)
#                         & (xc < cx_c + half)
#                         & (yc >= cy_c - half)
#                         & (yc < cy_c + half),
#                         axis=1,
#                     ).astype(np.float64)
#                 counts[r0:r1] = acc
#             X[rows, gi] = counts
#         if idx % 10 == 0:
#             print(f"Processed {idx} out of {len(genes)} genes!")

#     if assign_to == "X":
#         spots.X = X
#     else:
#         spots.layers[assign_to] = X
#     return spots

In [ ]:
# Read granules
granule_adata = sc.read_h5ad(data_path + "granule_adata_tsne.h5ad")

# Adjust coordinates
granule_adata.obs["global_x"] = granule_adata.obs["global_x_adjusted"].copy()
granule_adata.obs["global_y"] = granule_adata.obs["global_y_adjusted"].copy()
mask = granule_adata.obs["batch"] == "MERSCOPE_WT_1"
granule_adata.obs.loc[mask, "global_x"] = (6250 - granule_adata.obs.loc[mask, "global_x"])

# Read granule subtype labels
granule_subtype_df = pd.read_parquet(data_path + "granule_subtype_labels_granule_adata_tsne.parquet")
cols_keep = ["sample", "granule_id", "granule_subtype_kmeans", "granule_subtype_manual", "granule_subtype_manual_simple"]
granule_subtype_df = granule_subtype_df[cols_keep].drop_duplicates(["sample", "granule_id"])

# Merge granule subtype labels
granule_adata.obs = granule_adata.obs.reset_index(names="obs_name")
granule_adata.obs = granule_adata.obs.merge(granule_subtype_df,
                                            left_on=["batch", "granule_id"],
                                            right_on=["sample", "granule_id"],
                                            how="left",
                                            validate="one_to_one").set_index("obs_name")
if "sample" in granule_adata.obs.columns:
    granule_adata.obs = granule_adata.obs.drop(columns=["sample"])

# Convert granule subtype labels to category
granule_adata.obs["granule_subtype_kmeans"] = granule_adata.obs["granule_subtype_kmeans"].astype("category")
granule_adata.obs["granule_subtype_manual"] = granule_adata.obs["granule_subtype_manual"].astype("category")
granule_adata.obs["granule_subtype"] = granule_adata.obs["granule_subtype_manual_simple"].astype("category")

In [ ]:
# # Read spots
# spots_wt = sc.read_h5ad(data_path_wt + "spots.h5ad")
# spots_ad = sc.read_h5ad(data_path_ad + "spots.h5ad")

# # Adjust coordinates
# spots_wt.obs["global_x"] = spots_wt.obs["global_y_new"].copy()
# spots_wt.obs["global_y"] = spots_wt.obs["global_x_new"].copy()

# spots_ad.obs["global_x"] = spots_ad.obs["global_x_new"].copy()
# spots_ad.obs["global_y"] = spots_ad.obs["global_y_new"].copy()
# spots_ad.obs["global_x"] += 12000
# spots_ad.obs["global_y"] += 7200

# spots_dict = {"MERSCOPE_WT_1": spots_wt, "MERSCOPE_AD_1": spots_ad}
# spots = anndata.concat(spots_dict, axis = 0, merge = "same", label = "batch")
# spots = spots[spots.obs["brain_area"] != "Unknown"].copy()

In [ ]:
# # Read transcripts
# transcripts = pd.read_parquet(data_path + "neuropil_subdomains_transcripts.parquet")
# transcripts_extrasomatic = transcripts[transcripts["overlaps_nucleus"] == 0]

In [ ]:
# # Fill spot extrasomatic expression
# spots = fill_spot_expression(spots, transcripts_extrasomatic, grid_len=50.0, assign_to="extrasomatic_transcripts")

# # Fill spot granule expression
# embeddings, embeddings_features, aux_features, spot_granule_expression, _ = spot_embedding(
#     spots=spots,
#     granule_adata=granule_adata,
#     spot_loc_key=("global_x", "global_y"),
#     spot_width=50,
#     spot_height=50,
#     granule_loc_key=("global_x", "global_y"),
#     granule_subtype_key="granule_subtype_kmeans",
#     subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
#     granule_count_layer="counts",
#     include_soma_features=False,
#     smoothing=False
# )

# spots.layers["granule_expression"] = spot_granule_expression
# spots.layers["ambient_expression"] = np.maximum(spots.layers["extrasomatic_transcripts"] - spots.layers["granule_expression"], 0)

# # Save spots
# spots.write_h5ad(output_path + "spots_ambient.h5ad")

In [ ]:
# Read spots
spots = sc.read_h5ad(output_path + "spots_ambient.h5ad")
spots

In [ ]:
# Granule markers
gnl_genes = ["Camk2a", "Cplx2", "Slc17a7", "Ddn", "Syp", "Map1a", "Shank1", "Syn1", "Gria1", "Gria2", "Cyfip2", "Vamp2", "Bsn", "Slc32a1", "Nfasc", "Syt1", "Tubb3", "Nav1", "Shank3", "Mapt"]

In [ ]:
# Regression test: log spot subtype count ~ WT/AD + ambient(marker genes)
# - Per brain region, per granule subtype
# - Spot-level subtype counts computed IDENTICALLY to benchmark_subtyping.ipynb (containment-based counting)
# - AD subtype counts adjusted by capture efficiency coefficient (same as benchmark_subtyping.ipynb)
# - Bonferroni correction across brain regions within each subtype

CAPTURE_EFFICIENCY_COEF = 0.818691
GRID_LEN = 50.0

# Bonferroni within each subtype across tested brain regions
def bonferroni_within_group(p_series: pd.Series) -> pd.Series:
    p = p_series.to_numpy(dtype=float)
    ok = np.isfinite(p)
    m = int(ok.sum())
    out = np.full_like(p, np.nan, dtype=float)
    if m > 0:
        out[ok] = np.minimum(p[ok] * m, 1.0)
    return pd.Series(out, index=p_series.index)


# Copied from benchmark_subtyping.ipynb (same logic)
def compute_subtype_per_spot_counts(
    granule_obs,
    spots_one,
    area_col="brain_area",
    subtype_col="granule_subtype",
    sample_col="sample",
    coord_keys=("global_x", "global_y"),
    grid_len=50,
):
    """Return DataFrame with one row per (sample, brain_area, subtype, spot): columns sample, brain_area, subtype, count."""
    if (
        area_col not in spots_one.obs.columns
        or coord_keys[0] not in spots_one.obs.columns
        or coord_keys[1] not in spots_one.obs.columns
    ):
        return pd.DataFrame()

    half = grid_len / 2
    rows = []
    for sample in granule_obs[sample_col].dropna().unique():
        go = granule_obs[granule_obs[sample_col] == sample].copy()
        xcol = "global_x" if "global_x" in go.columns else "sphere_x"
        ycol = "global_y" if "global_y" in go.columns else "sphere_y"
        if xcol not in go.columns or ycol not in go.columns:
            continue

        for area in spots_one.obs[area_col].dropna().unique():
            sp = spots_one.obs.loc[spots_one.obs[area_col] == area]
            if len(sp) == 0:
                continue

            for subtype in go[subtype_col].dropna().unique():
                gs = go.loc[go[subtype_col] == subtype]
                for _, srow in sp.iterrows():
                    x, y = srow[coord_keys[0]], srow[coord_keys[1]]
                    in_spot = (
                        (gs[xcol].values >= x - half)
                        & (gs[xcol].values < x + half)
                        & (gs[ycol].values >= y - half)
                        & (gs[ycol].values < y + half)
                    )
                    count = in_spot.sum()
                    rows.append(
                        {"sample": sample, "brain_area": area, "subtype": subtype, "count": count, "spot_id": srow.get("spot_id", None)}
                    )

            # Overall (all granules per spot)
            for _, srow in sp.iterrows():
                x, y = srow[coord_keys[0]], srow[coord_keys[1]]
                in_spot = (
                    (go[xcol].values >= x - half)
                    & (go[xcol].values < x + half)
                    & (go[ycol].values >= y - half)
                    & (go[ycol].values < y + half)
                )
                count = in_spot.sum()
                rows.append(
                    {"sample": sample, "brain_area": area, "subtype": "overall", "count": count, "spot_id": srow.get("spot_id", None)}
                )

    return pd.DataFrame(rows)

In [ ]:
# ------------------------------
# Build per-spot subtype counts (identical to benchmark_subtyping.ipynb)
# ------------------------------
spots = spots[spots.obs["brain_area"].notna()].copy()
spots = spots[spots.obs["brain_area"] != "Unknown"].copy()

granule_obs = granule_adata.obs.copy()
granule_obs = granule_obs[granule_obs["granule_subtype"].notna()].copy()

# Split by sample/batch, but use the same 'sample' labels as benchmark_subtyping
spots_wt = spots[spots.obs["batch"] == "MERSCOPE_WT_1"].copy()
spots_ad = spots[spots.obs["batch"] == "MERSCOPE_AD_1"].copy()

gobs_wt = granule_obs[granule_obs["batch"] == "MERSCOPE_WT_1"].copy()
gobs_ad = granule_obs[granule_obs["batch"] == "MERSCOPE_AD_1"].copy()
gobs_wt["sample"] = "WT"
gobs_ad["sample"] = "AD"

per_spot_wt = compute_subtype_per_spot_counts(
    gobs_wt,
    spots_wt,
    subtype_col="granule_subtype",
    sample_col="sample",
    coord_keys=("global_x", "global_y"),
    grid_len=GRID_LEN,
)
per_spot_ad = compute_subtype_per_spot_counts(
    gobs_ad,
    spots_ad,
    subtype_col="granule_subtype",
    sample_col="sample",
    coord_keys=("global_x", "global_y"),
    grid_len=GRID_LEN,
)

# Adjust AD counts for capture efficiency (same as benchmark_subtyping)
per_spot_ad = per_spot_ad.copy()
per_spot_ad["count"] = per_spot_ad["count"].astype(float) / CAPTURE_EFFICIENCY_COEF

per_spot_all = pd.concat([per_spot_wt, per_spot_ad], ignore_index=True)
per_spot_all["count"] = per_spot_all["count"].astype(float)

subtypes = sorted(per_spot_all["subtype"].dropna().unique().tolist())

In [ ]:
# ------------------------------
# Ambient marker covariate (spot-level, using only gnl_genes)
# ------------------------------
# IMPORTANT: `spot_id` is not globally unique across WT vs AD, so we key by (batch, spot_id)
marker_genes_present = [g for g in gnl_genes if g in spots.var_names]
if len(marker_genes_present) == 0:
    raise ValueError("None of gnl_genes found in spots.var_names")
marker_idx = [int(np.where(spots.var_names == g)[0][0]) for g in marker_genes_present]

ambient = np.asarray(spots.layers["ambient_expression"], dtype=float)
ambient_marker_sum = np.clip(ambient[:, marker_idx], 0.0, None).sum(axis=1)
ad_mask = spots.obs["batch"] == "MERSCOPE_AD_1"
ambient_marker_sum[ad_mask] = ambient_marker_sum[ad_mask] / CAPTURE_EFFICIENCY_COEF
ambient_marker_cov = np.log1p(ambient_marker_sum)

ambient_df = pd.DataFrame({
    "spot_id": spots.obs["spot_id"].to_numpy(),
    "batch": spots.obs["batch"].to_numpy(),
    "ambient_marker_cov": ambient_marker_cov,
})

per_spot_all = per_spot_all.copy()
per_spot_all["batch"] = per_spot_all["sample"].replace({"WT": "MERSCOPE_WT_1", "AD": "MERSCOPE_AD_1"})

per_spot_all = per_spot_all.merge(
    ambient_df,
    on=["batch", "spot_id"],
    how="left",
    validate="many_to_one",
)

In [ ]:
# ------------------------------
# Regression per (subtype, brain_area)
# ------------------------------
# Full model: log1p(count) ~ AD_indicator + ambient_marker_cov
results = []

for subtype in subtypes:
    df_sub = per_spot_all[per_spot_all["subtype"] == subtype].copy()

    for area in sorted(df_sub["brain_area"].dropna().unique().tolist()):
        dfa = df_sub[df_sub["brain_area"] == area].copy()

        y = np.log1p(dfa["count"].to_numpy(dtype=float))
        x_ad = (dfa["sample"] == "AD").astype(float).to_numpy()
        x_amb = dfa["ambient_marker_cov"].to_numpy(dtype=float)

        X = np.column_stack([x_ad, x_amb])
        X = sm.add_constant(X, has_constant="add")

        n_wt = int((dfa["sample"] == "WT").sum())
        n_ad = int((dfa["sample"] == "AD").sum())

        if n_wt < 2 or n_ad < 2:
            beta_ad = np.nan
            se_ad = np.nan
            p_ad = np.nan
            beta_amb = np.nan
            se_amb = np.nan
            p_amb = np.nan
        else:
            # No robust covariance: matches pooled-variance inference underlying classic two-sample t-test
            fit = sm.OLS(y, X).fit()
            beta_ad = float(fit.params[1])
            se_ad = float(fit.bse[1])
            p_ad = float(fit.pvalues[1])
            beta_amb = float(fit.params[2])
            se_amb = float(fit.bse[2])
            p_amb = float(fit.pvalues[2])

        results.append({
            "subtype": subtype,
            "brain_area": area,
            "n_spots_wt": n_wt,
            "n_spots_ad": n_ad,
            "mean_count_wt": float(dfa.loc[dfa["sample"] == "WT", "count"].mean()),
            "mean_count_ad_adj": float(dfa.loc[dfa["sample"] == "AD", "count"].mean()),
            "beta_AD": beta_ad,
            "se_AD": se_ad,
            "p_val_AD": p_ad,
            "beta_ambient": beta_amb,
            "se_ambient": se_amb,
            "p_val_ambient": p_amb,
        })

res_df = pd.DataFrame(results)
res_df["p_bonf_AD"] = res_df.groupby("subtype", group_keys=False)["p_val_AD"].apply(bonferroni_within_group)
res_df["p_bonf_ambient"] = res_df.groupby("subtype", group_keys=False)["p_val_ambient"].apply(bonferroni_within_group)

res_df["p_val_AD_star"] = res_df["p_val_AD"].apply(lambda p: "" if pd.isna(p) else p_val_to_star(p))
res_df["p_val_ambient_star"] = res_df["p_val_ambient"].apply(lambda p: "" if pd.isna(p) else p_val_to_star(p))
res_df["p_bonf_AD_star"] = res_df["p_bonf_AD"].apply(lambda p: "" if pd.isna(p) else p_val_to_star(p))
res_df["p_bonf_ambient_star"] = res_df["p_bonf_ambient"].apply(lambda p: "" if pd.isna(p) else p_val_to_star(p))

# Save full model
res_df.to_csv(output_path + "subtype_density_per_region_ambient_regression.csv", index=False)


# ------------------------------
# Degenerate model (no ambient covariate): log1p(count) ~ AD_indicator
# ------------------------------
results_degenerate = []

for subtype in subtypes:
    df_sub = per_spot_all[per_spot_all["subtype"] == subtype].copy()

    for area in sorted(df_sub["brain_area"].dropna().unique().tolist()):
        dfa = df_sub[df_sub["brain_area"] == area].copy()

        y = np.log1p(dfa["count"].to_numpy(dtype=float))
        x_ad = (dfa["sample"] == "AD").astype(float).to_numpy()

        X = sm.add_constant(x_ad, has_constant="add")

        n_wt = int((dfa["sample"] == "WT").sum())
        n_ad = int((dfa["sample"] == "AD").sum())

        if n_wt < 2 or n_ad < 2:
            beta_ad = np.nan
            se_ad = np.nan
            p_ad = np.nan
        else:
            fit = sm.OLS(y, X).fit()
            beta_ad = float(fit.params[1])
            se_ad = float(fit.bse[1])
            p_ad = float(fit.pvalues[1])

        results_degenerate.append({
            "subtype": subtype,
            "brain_area": area,
            "n_spots_wt": n_wt,
            "n_spots_ad": n_ad,
            "mean_count_wt": float(dfa.loc[dfa["sample"] == "WT", "count"].mean()),
            "mean_count_ad_adj": float(dfa.loc[dfa["sample"] == "AD", "count"].mean()),
            "beta_AD": beta_ad,
            "se_AD": se_ad,
            "p_val_AD": p_ad,
        })

res_df_deg = pd.DataFrame(results_degenerate)
res_df_deg["p_bonf_AD"] = res_df_deg.groupby("subtype", group_keys=False)["p_val_AD"].apply(bonferroni_within_group)
res_df_deg["p_val_AD_star"] = res_df_deg["p_val_AD"].apply(lambda p: "" if pd.isna(p) else p_val_to_star(p))
res_df_deg["p_bonf_AD_star"] = res_df_deg["p_bonf_AD"].apply(lambda p: "" if pd.isna(p) else p_val_to_star(p))

# Save degenerate model
res_df_deg.to_csv(output_path + "subtype_density_per_region_regression_noambient.csv", index=False)